# One graph, two Moxi entry points

This example uses a **local Python kernel on macOS ARM64** and the compiled FDG
binary. Follow [notebook/README.md](notebook/README.md) to build and launch JupyterLab.
The SVG is a static, settled snapshot. The native viewer is a separate process;
changing Python data after launch does not update it.

**Remote kernels:** the native window opens on the **kernel machine**, not in your
browser or on your laptop. SVG text travels through Jupyter and renders inline.
Actual remote-kernel operation has not been tested. This is not an inline widget.


In [ ]:
from pathlib import Path
import importlib.metadata
import json
import platform
import subprocess
import sys
import tomllib
from IPython.display import SVG, display

# Jupyter starts here, in examples/fdg/, or at the repository root.
here = Path.cwd().resolve()
candidates = (here / "examples" / "fdg", here, *here.parents)
example = next((p for p in candidates if (p / "moxi_fdg.py").is_file()), None)
if example is None:
    raise RuntimeError("Start JupyterLab in the repository root or examples/fdg/.")
sys.path.insert(0, str(example))
from moxi_fdg import render_svg, view_graph

root = example.parents[1]
manifest = tomllib.loads((root / "pixi.toml").read_text())
versions = {
    "Python": platform.python_version(),
    "JupyterLab": importlib.metadata.version("jupyterlab"),
    "platform": platform.platform(),
    "Moxi": manifest["workspace"]["version"],
    "binary source revision": (example / "dist" / "moxi-revision.txt").read_text().strip(),
}
print(json.dumps(versions, indent=2))


## Define once

Both helpers below consume this same five-node graph. Labels also exercise Unicode
and XML escaping. The solver and graph interpretation belong to the FDG example.


In [ ]:
graph = {
    "nodes": [
        {"id": "Data", "group": 0},
        {"id": "Python", "group": 1},
        {"id": "Mojo", "group": 2},
        {"id": "Moxi", "group": 2},
        {"id": "SVG & μ", "group": 3},
    ],
    "links": [
        {"source": "Data", "target": "Python", "value": 1},
        {"source": "Python", "target": "Mojo", "value": 1},
        {"source": "Mojo", "target": "Moxi", "value": 1},
        {"source": "Moxi", "target": "SVG & μ", "value": 1},
        {"source": "Data", "target": "SVG & μ", "value": 1},
    ],
}
print("One graph: 5 nodes, 5 links")


In [ ]:
svg = render_svg(graph)
display(SVG(svg))


## Launch without blocking the kernel

Run this cell twice to open two independent viewers. Each returned `Popen` is
retained for cleanup. Run the next cell while the windows remain open.
Native diagnostics inherit kernel stdout/stderr; a failed nonblocking launch is
reported through `poll()`/`wait()`, and may appear in the Jupyter server log.


In [ ]:
viewers = globals().get("viewers", [])
viewer = view_graph(graph)
viewers.append(viewer)
print("Launch returned; retained viewer handles:", len(viewers))


In [ ]:
answer = sum(i * i for i in range(100_000))
assert answer == 333328333350000
print("Kernel computed:", answer)
print("Viewer running:", viewer.poll() is None)


## An actionable error

The compiled parser rejects an unknown endpoint. Read `CalledProcessError.stderr`
for the actual diagnostic; the repair is to use a node ID present in `nodes`.
The original graph stays unchanged. A missing executable instead raises
`FileNotFoundError` with build/environment-variable instructions.


In [ ]:
invalid_graph = {**graph, "links": [{"source": "Data", "target": "missing", "value": 1}]}
try:
    render_svg(invalid_graph)
except subprocess.CalledProcessError as error:
    print("Viewer rejected the graph:")
    print(error.stderr.decode("utf-8", errors="replace").strip())
    print("Repair: replace target 'missing' with an existing node ID, such as 'Python'.")
else:
    raise AssertionError("The invalid endpoint should have been rejected")


## Close and reopen

Close the native windows normally, then run this cell to reap their processes and
open the valid graph again. For unattended **Run All**, cleanup terminates any
windows you left open. Closing a window normally returns exit code 0; termination
has a different exit code. Each launch starts fresh and does not retain UI state.


In [ ]:
def close_viewers():
    for process in viewers:
        if process.poll() is None:
            process.terminate()
        process.wait(timeout=10)
    viewers.clear()

close_viewers()
viewer = view_graph(graph)
viewers.append(viewer)
print("Reopened the original graph")


## Clean up before restarting

Run cleanup before **Kernel → Restart Kernel and Run All Cells**. Restart destroys
Python handles; these independent native processes are not automatically owned or
closed by Jupyter. If you restart early, close their windows yourself. Restart and
Run All reconstructs the graph and SVG; final cleanup leaves no viewer running.
Outputs are intentionally cleared in the tracked notebook.


In [ ]:
close_viewers()
print("All retained viewers closed. Ready for kernel restart and Run All.")
